# Pairs Trading EDA

Exploratory data analysis for statistical arbitrage pairs trading.

**Universe:** KO, PEP, XOM, CVX, GLD, SLV  
**Period:** 2018-01-01 to 2025-01-01

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd

from sarb.backtest.walkforward import walkforward_pairs_backtest
from sarb.data.ingest import load_yfinance_prices
from sarb.features.spread import compute_spread, fit_hedge_ratio, rolling_zscore
from sarb.metrics.performance import cagr, max_drawdown, performance_summary, sharpe
from sarb.stats.cointegration import (
    engle_granger_adf_pvalue,
    engle_granger_test,
    estimate_half_life,
)
from sarb.viz.charts import (
    plot_correlation_heatmap,
    plot_drawdown,
    plot_equity_curve,
    plot_rolling_beta,
    plot_spread,
    plot_tearsheet,
)

## 1. Load Price Data

In [ ]:
tickers = ["KO", "PEP", "XOM", "CVX", "GLD", "SLV"]
prices = load_yfinance_prices(tickers, "2018-01-01", "2025-01-01", field="Adj Close")
prices = prices.dropna(axis=1, how="any")
print(f"Shape: {prices.shape}")
prices.tail()

## 2. Normalized Price Series

In [ ]:
normalized = prices / prices.iloc[0] * 100
ax = normalized.plot(figsize=(12, 5), title="Normalized Prices (base=100)")
ax.set_ylabel("Price")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left")

## 3. Return Correlation Heatmap

In [ ]:
returns = prices.pct_change().dropna()
fig = plot_correlation_heatmap(returns, title="Daily Return Correlations")
fig

## 4. Spread Analysis (KO/PEP)

In [ ]:
y_tkr, x_tkr = "KO", "PEP"

# Fit on first 70% of data
n_train = int(len(prices) * 0.7)
train_px = prices.iloc[:n_train]

alpha, beta = fit_hedge_ratio(train_px[y_tkr], train_px[x_tkr])
print(f"Hedge ratio: {y_tkr} = {alpha:.4f} + {beta:.4f} * {x_tkr}")

spread = compute_spread(prices[y_tkr], prices[x_tkr], alpha, beta)
z = rolling_zscore(spread, 60)

fig = plot_spread(spread, z, entry_z=2.0, exit_z=0.5, title=f"{y_tkr}/{x_tkr} Spread & Z-Score")
fig

## 5. Cointegration Diagnostics

`naive_ADF_p` runs an ordinary ADF test on OLS residuals. That is anti-conservative: OLS picks
the hedge ratio that makes the residual look most stationary, so Dickey-Fuller critical values
reject too often. `EG_p` uses MacKinnon's Engle-Granger critical values, which account for the
estimated hedge ratio. The pipeline uses `EG_p`.

In [ ]:
import itertools

results = []
for y, x in itertools.combinations(list(prices.columns), 2):
    eg = engle_granger_test(train_px[y], train_px[x])  # MacKinnon cointegration p-value
    sp = compute_spread(train_px[y], train_px[x], eg.alpha, eg.beta)
    naive = engle_granger_adf_pvalue(sp)  # plain ADF on fitted residuals
    results.append(
        {
            "Y": y,
            "X": x,
            "beta": eg.beta,
            "corr": returns[y].corr(returns[x]),
            "EG_p": eg.pvalue,
            "naive_ADF_p": naive,
            "half_life": estimate_half_life(sp),
        }
    )

diag = pd.DataFrame(results).sort_values("EG_p")
diag.round({"beta": 3, "corr": 3, "EG_p": 4, "naive_ADF_p": 4, "half_life": 1})

## 6. Walk-Forward Backtest

In [ ]:
bt = walkforward_pairs_backtest(
    prices=prices[[y_tkr, x_tkr]].dropna(),
    y=y_tkr,
    x=x_tkr,
    train_lookback=504,
    z_lookback=60,
    entry_z=2.0,
    exit_z=0.5,
    fee_bps=1.0,
    slippage_bps=0.5,
)

# Out-of-sample window only
test_bt = bt.iloc[n_train:]
pd.Series(performance_summary(test_bt)).to_frame(f"{y_tkr}/{x_tkr} OOS")

In [ ]:
fig = plot_tearsheet(test_bt["ret_net"], title=f"{y_tkr}/{x_tkr} walk-forward (OOS)")
fig

In [ ]:
# beta is a *share* ratio from a price-level regression; the dollar hedge is beta * Px / Py
dollar_ratio = (bt["beta"] * prices[x_tkr] / prices[y_tkr]).dropna()
print(f"median share ratio:  {bt['beta'].median():.2f}")
print(f"median dollar ratio: {dollar_ratio.median():.2f}")

In [ ]:
fig = plot_rolling_beta(bt["beta"].dropna(), title=f"{y_tkr}/{x_tkr} Rolling Hedge Ratio")
fig

## Summary

This notebook walked through the core pipeline:
1. **Data loading** from Yahoo Finance (cached under `data/cache/`)
2. **Correlation analysis** to find candidate pairs
3. **Spread modeling** with an OLS hedge ratio and z-score normalization
4. **Cointegration testing** with Engle-Granger p-values (and why naive ADF over-rejects)
5. **Walk-forward backtesting** with a daily hedge-ratio refit, scored out of sample only

For universe-level research use `sarb scan` (FDR-controlled selection with deflated Sharpe ratios)
and `sarb portfolio` (quarterly walk-forward multi-pair portfolio).